### 2. Import necessary libraries

In [191]:
try:
    import datasets, evaluate, accelerate
    import gradio as gr
except ModuleNotFoundError:
    !pip install -U datasets evaluate accelerate gradio
    import datasets, evaluate, accelerate
    import gradio as gr
import random
import numpy as np
import pandas as pd
import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using torch version: {torch.__version__}")
print(f"Using datasets version: {datasets.__version__}")

Using transformers version: 4.45.2
Using torch version: 2.4.1
Using datasets version: 3.0.1


## 3. Getting a dataset
Building food not food text classification model: need food not food text dataset.

In [192]:
from datasets import load_dataset

# Load the dataset from Hugging Face Hub
dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")

#Inspect the dataset
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [193]:
# What features are there?
dataset.column_names

{'train': ['text', 'label']}

In [194]:
# Access the training split
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [195]:
# How about we check out a single sample?
# We can do so with indexing.
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Inspect random samples

In [196]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)
print(random_indexs)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text,label in zip(random_samples["text"], random_samples["label"]):
    print(f"Text: {text} | Label: {label}")

[163, 28, 6, 189, 70]
[INFO] Random samples from dataset:

Text: Set of spatulas kept in a holder | Label: not_food
Text: Mouthwatering paneer tikka masala, featuring juicy paneer in a rich tomato-based sauce, garnished with fresh coriander leaves. | Label: food
Text: Pair of reading glasses left open on a book | Label: not_food
Text: Set of board games stacked on a shelf | Label: not_food
Text: Two handfuls of bananas in a fruit bowl with grapes on the side, the fruit bowl is blue | Label: food


In [197]:
# Get unique label values
dataset["train"].unique("label")

['food', 'not_food']

In [198]:
# Check the count of each label
from collections import Counter
Counter(dataset["train"]["label"])

Counter({'food': 125, 'not_food': 125})

In [199]:
# Turn our dataset into a DataFrame and get a random sample
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
142,A slice of pizza with a generous amount of shr...,food
6,Pair of reading glasses left open on a book,not_food
97,Telescope positioned on a balcony,not_food
60,A close-up of a family playing a board game wi...,not_food
112,Rich and spicy lamb rogan josh with yogurt gar...,food
181,"A steaming bowl of fiery chicken curry, infuse...",food
197,"Pizza with a stuffed crust, oozing with cheese",food


In [200]:
food_not_food_df["label"].value_counts()

label
food        125
not_food    125
Name: count, dtype: int64

## 4. Preparing data for text classification

1. Tokenization - turning our text into a numerical representation (machines prefer numbers rather than words), for example, {"a": 0, "b": 1, "c": 2...}.
2. Creating a train/test split - right now our data is in a training split only but we'll create a test set to evaluate our model's performance.

In [201]:
# Create a mapping for labels to numeric value
id2label = {0: "not_food", 1: "food"}
label2id = {"not_food": 0, "food": 1}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [202]:
# Create mappings programmatically from dataset
id2label = {idx: label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id = {label: idx for idx, label in id2label.items()}
print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [203]:
id2label = {}
for idx, label in enumerate(dataset["train"].unique("label")[::-1]):
    print(idx, label)
    id2label[idx] = label

0 not_food
1 food


In [204]:
# Turn labels into 0 or 1
def map_labels_to_number(example):
    example["label"] = label2id[example["label"]]
    return example

example_sample = {"text": "This is a sentence about my favourite food: honey", "label": "food"}

# Test our function
print(map_labels_to_number(example_sample))

{'text': 'This is a sentence about my favourite food: honey', 'label': 1}


In [205]:
# Map our dataset labels to numbers (the whole thing)
# We do this with dataset.map()
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}

In [206]:
# Shuffle data and look at 5 more random samples
dataset.shuffle()[:5]

{'text': ['Set of oven mitts hanging on a hook',
  'Set of cookie cutters collected in a jar',
  'Pizza with a dessert twist, featuring a sweet Nutella base and fresh strawberries on top',
  'Set of binoculars placed on a table',
  'Two handfuls of bananas in a fruit bowl with grapes on the side, the fruit bowl is blue'],
 'label': [0, 0, 1, 0, 1]}

### Split the dataset into training and test sets
* Train set = model will learn patterns on this dataset
* Validation set (optional) = We can tune our model's hyperparameters on this set
* Test set = model will evaluate patterns on this dataset

In [207]:
# Split our dataset into train/test splits
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 50
    })
})

In [208]:
random_idx_train = random.randint(0, len(dataset["train"]))
random_sample_train = dataset["train"][random_idx_train]
random_sample_train

{'text': 'Set of dumbbells stacked in a gym', 'label': 0}

In [209]:
random_idx_test = random.randint(0, len(dataset["test"]))
random_sample_test = dataset["test"][random_idx_test]
random_sample_test

{'text': 'Two handfuls of bananas in a fruit bowl with grapes on the side, the fruit bowl is blue',
 'label': 1}

### Tokenizing our text data (Turning text into numbers)
The premise of tokenization is to turn words into numbers.
Eg. "I love pizza!" -> [101, 1045, 2293, 10733, 102]

-

The `transformers` library has in-built support for Hugging Face tokenizers.
And the class `transformers.AutoTokenizer` helps pair a model to a tokenizer.

In [210]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
                                          use_fast=True) # uses fast tokenization (backed by tokenziers library and implemented in Rust) by default, if not available will default to Python implementation

tokenizer

DistilBertTokenizerFast(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [211]:
# Test out the tokenizer
tokenizer("I love pizza")

{'input_ids': [101, 1045, 2293, 10733, 102], 'attention_mask': [1, 1, 1, 1, 1]}

* `input_ids` = our text turned into numbers
* `attention_mask` = Whether or not to pay attention to certain tokens (1 = yes pay attention, 0 = no don't pay attention)

In [212]:
# Get the length of our tokenizer vocab
length_of_tokenizer_vocab = len(tokenizer.vocab)
print(f"[INFO] Number of items in our tokenizer vocab: {length_of_tokenizer_vocab}")

# Get the maximum sequence length the tokenizer can handle
max_tokenizer_input_sequence_length = tokenizer.model_max_length
print(f"[INFO] Max tokenizer input sequence length: {max_tokenizer_input_sequence_length}")

[INFO] Number of items in our tokenizer vocab: 30522
[INFO] Max tokenizer input sequence length: 512


In [213]:
# Does 'daniel' occur in the vocab
tokenizer.vocab["daniel"]

3817

In [214]:
tokenizer("adnan")

{'input_ids': [101, 4748, 7229, 102], 'attention_mask': [1, 1, 1, 1]}

In [215]:
tokenizer.convert_ids_to_tokens(tokenizer("adnan").input_ids)

['[CLS]', 'ad', '##nan', '[SEP]']

In [216]:
# Try to tokenize an emoji
tokenizer.convert_ids_to_tokens(tokenizer("🍕").input_ids)

['[CLS]', '[UNK]', '[SEP]']

In [217]:
# Get the first 5 items in the tokenizer vocab
sorted(tokenizer.vocab.items())[:5]

[('!', 999), ('"', 1000), ('#', 1001), ('##!', 29612), ('##"', 29613)]

In [218]:
random.sample(sorted(tokenizer.vocab.items()), k=5)

[('##vies', 25929),
 ('responsibility', 5368),
 ('##pm', 9737),
 ('persona', 16115),
 ('rhythm', 6348)]

### Making a preprocessing function to tokenize text
Want to make it easy to go from sample -> tokenized_sample

In [219]:
def tokenize_text(examples):
    """
    Tokenize given example text and return the tokenized text.
    """
    return tokenizer(examples["text"],
                     padding=True, # pad short sequences to longest sequence in the batch (e.g. if sample length = 100, sample will be padded to 512 or longest sample in batch)
                     truncation=True) # truncate long sequences to the maximum length the model can handle (e.g. if sample length = 1000, model length = 512, sample will be shortened to 512)

In [220]:
example_sample_2 = {"text": "I love pizza", "label": 1}
# Test the function
tokenize_text(example_sample_2)

{'input_ids': [101, 1045, 2293, 10733, 102], 'attention_mask': [1, 1, 1, 1, 1]}

In [221]:
# Check whether truncation is working or not
long_text = "I love pizza " * 1000
len(long_text)

13000

In [222]:
tokenized_long_text = tokenize_text({"text": long_text, "label": 1})
len(tokenized_long_text["input_ids"])

512

In [223]:
# Map our tokenize text function to the dataset
tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True, # Set batched=True to tokenize across batches of samples at a time rather than one at a time.
                                batch_size=1000
                                )
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50
    })
})

In [224]:
# Get two samples from the tokenized datasets
train_tokenized_sample = tokenized_dataset["train"][0]
test_tokenized_sample = tokenized_dataset["test"][0]
for key in train_tokenized_sample.keys():
    print(f"[INFO] key: {key}")
    print(f"Train Sample: {train_tokenized_sample[key]}")
    print(f"Test Sample: {test_tokenized_sample[key]}")
    print()

[INFO] key: text
Train Sample: Set of headphones placed on a desk
Test Sample: A slice of pepperoni pizza with a layer of melted cheese

[INFO] key: label
Train Sample: 0
Test Sample: 1

[INFO] key: input_ids
Train Sample: [101, 2275, 1997, 2132, 19093, 2872, 2006, 1037, 4624, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test Sample: [101, 1037, 14704, 1997, 11565, 10698, 10733, 2007, 1037, 6741, 1997, 12501, 8808, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

[INFO] key: attention_mask
Train Sample: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test Sample: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]



### Setting up an evaluation metric
What we want to do: Use the evaluation metric to get a numerical idea of how our model is performing.

Some common evaluation metrics for classification:

- Accuracy (How many examples out of 100, did you get correct?)
- Precision
- Recall
- F1 Score

Evaluation metric is important because some projects may have an evaluation threshold you need to fulfill. 

In [225]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

In [226]:
def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
    """
    Computes the accuracy of a model by comparing the predictions and labels. 
    """
    predictions, labels = predictions_and_labels
    if len(predictions.shape) >= 2:
        predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [227]:
# Create example list of predictions and labels
example_predictions_all_correct = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
example_predictions_one_wrong = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])
example_labels = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

# Test the function
print(f"Accuracy when all predictions are correct: {compute_accuracy((example_predictions_all_correct, example_labels))}")
print(f"Accuracy when one prediction is wrong: {compute_accuracy((example_predictions_one_wrong, example_labels))}")

Accuracy when all predictions are correct: {'accuracy': 1.0}
Accuracy when one prediction is wrong: {'accuracy': 0.9}


### Setting up a model for training

1. ✅ Create and preprocess data.
2. Define the model we'd like use with transformers.AutoModelForSequenceClassification (or another similar model class).
3. Define training arguments (these are hyperparameters for our model) with `transformers.TrainingArguments`.
4. Pass TrainingArguments from 3 and target datasets to an instance of `transformers.Trainer`.
5. Train the model by calling `Trainer.train()`.
6. Save the model (to our local machine or to the Hugging Face Hub).
7. Evaluate the trained model by making and inspecting predctions on the test data.
8. Turn the model into a shareable demo.

In [228]:
from transformers import AutoModelForSequenceClassification

# We are instantiating the base model
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="distilbert/distilbert-base-uncased", # Base Model
    num_labels=2, # Classify into food/not_food
    id2label=id2label,
    label2id=label2id
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [229]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

#### Our model is comprised of following parts:
1. `embeddings` - embeddings are a form of learned representation of tokens. So if tokensare a direct mapping from token to number, embeddings are a learned vector representation.

2. `transformer` - Our model architecture backbone, this has discovered patterns/relationships in the embeddings.

3. `classifier` - We need to customize this layer to suit our problem.

* Note: If you get input errors from passing a sample to a model, make sure the sample you pass to your model is formatting in the same way your model was trained on. For example, if your model used a specific tokenizer, make sure to tokenize your text before passing it to the model. 

### Count the parameters in our model

* Weights/parameters = small numeric opportunities for a model to learn patterns in data.

* We want to tune weights/parameters in our model that already been pre-learned at a certain datasets to our own problem.

In [230]:
def count_params(model):
    """
    Count the parameters of a PyTorch model. 
    param.requires_grad -> True -> It's going to be updated during training
    if False then it won't be updated during training. 
    """
    trainable_parameters = sum(param.numel() for param in model.parameters() if param.requires_grad)
    total_parameters = sum(param.numel() for param in model.parameters())
    return {"trainable_parameters": trainable_parameters, "total_parameters": total_parameters}

In [231]:
count_params(model)

{'trainable_parameters': 66955010, 'total_parameters': 66955010}

### Create a directory for saving models

In [232]:
# Create model output directory
from pathlib import Path

# Create models dir
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# Create model save name
model_save_name = "learn_hf_food_not_food_text_classifier-distilbert-base-uncased"

# Create model save path
model_save_dir = Path(models_dir, model_save_name)

model_save_dir

PosixPath('models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased')

### Setting up training arguments (Hyperparameters) with TrainingArguments

* Define training arguments for training our model `transformers.TrainingArguments`

    * Hyperparameters = Settings on your model that you can adjust
    * Parameters = Weights/Patters in the model that get updated automatically. 

In [233]:
from transformers import TrainingArguments
from dotenv import load_dotenv
import os

load_dotenv()

print(f"[INFO] Saving model checkpoints to: {model_save_dir}")

# Create training arguments
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10, # Laps over the data that our model is going to do
    eval_strategy="epoch", # was previously "evaluation_strategy"
    save_strategy="epoch",
    save_total_limit=3, # limit the total amount of save checkpoints (so we don't save num_epochs checkpoints)
    use_cpu=False, # set to False by default, will use CUDA GPU or MPS device if available
    seed=42, # set to 42 by default for reproducibility
    load_best_model_at_end=True, # load the best model when finished training
    logging_strategy="epoch", # log training results every epoch
    report_to="none", # optional: log experiments to Weights & Biases/other similar experimenting tracking services (we'll turn this off for now) 
    # push_to_hub=True # optional: automatically upload the model to the Hub (we'll do this manually later on)
    hub_token=os.getenv("HF_TOKEN"), # optional: add your Hugging Face Hub token to push to the Hub (will default to huggingface-cli login)
    hub_private_repo=False # optional: make the uploaded model private (defaults to False)
)

[INFO] Saving model checkpoints to: models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased


In [234]:
# Optional: Print out training_args to inspect (warning, it is quite a long output)
training_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
evaluation_s

### Setting up an instance of Trainer - 4 

1. ✅ Create and preprocess data.
2. ✅ Define the model we'd like use with transformers.AutoModelForSequenceClassification (or another similar model class).
3. ✅ Define training arguments (these are hyperparameters for our model) with `transformers.TrainingArguments`.
4. Pass TrainingArguments from 3 and target datasets to an instance of `transformers.Trainer`.
5. Train the model by calling `Trainer.train()`.
6. Save the model (to our local machine or to the Hugging Face Hub).
7. Evaluate the trained model by making and inspecting predctions on the test data.
8. Turn the model into a shareable demo.

In [235]:
from transformers import Trainer

#Setup Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_accuracy
)
trainer

### Train the model by calling `Trainer.train()` - 5

1. ✅ Create and preprocess data.
2. ✅ Define the model we'd like use with transformers.AutoModelForSequenceClassification (or another similar model class).
3. ✅ Define training arguments (these are hyperparameters for our model) with `transformers.TrainingArguments`.
4. ✅ Pass TrainingArguments from 3 and target datasets to an instance of `transformers.Trainer`.
5. Train the model by calling `Trainer.train()`.
6. Save the model (to our local machine or to the Hugging Face Hub).
7. Evaluate the trained model by making and inspecting predctions on the test data.
8. Turn the model into a shareable demo.

In [236]:
result = trainer.train()

 10%|█         | 7/70 [00:02<00:14,  4.29it/s]
                                              

 10%|█         | 7/70 [00:02<00:14,  4.29it/s]


{'loss': 0.3173, 'grad_norm': 0.9694774150848389, 'learning_rate': 9e-05, 'epoch': 1.0}





                                              

                                        


 10%|█         | 7/70 [00:02<00:14,  4.29it/s]






{'eval_loss': 0.04073445498943329, 'eval_accuracy': 1.0, 'eval_runtime': 0.1108, 'eval_samples_per_second': 451.395, 'eval_steps_per_second': 18.056, 'epoch': 1.0}


 20%|██        | 14/70 [00:05<00:14,  3.91it/s]
                                               

 20%|██        | 14/70 [00:05<00:14,  3.91it/s]


...

{'loss': 0.0007, 'grad_norm': 0.012042270042002201, 'learning_rate': 0.0, 'epoch': 10.0}





                                               

                                        


100%|██████████| 70/70 [00:31<00:00,  3.90it/s]






{'eval_loss': 0.0005715943989343941, 'eval_accuracy': 1.0, 'eval_runtime': 0.129, 'eval_samples_per_second': 387.618, 'eval_steps_per_second': 15.505, 'epoch': 10.0}



                                               

100%|██████████| 70/70 [00:33<00:00,  2.11it/s]

{'train_runtime': 33.1545, 'train_samples_per_second': 60.324, 'train_steps_per_second': 2.111, 'train_loss': 0.03469283777023, 'epoch': 10.0}


In [237]:
result.metrics

{'train_runtime': 33.1545,
 'train_samples_per_second': 60.324,
 'train_steps_per_second': 2.111,
 'total_flos': 18110777160000.0,
 'train_loss': 0.03469283777023,
 'epoch': 10.0}

In [238]:
# Inspect training metrics
for key, value in result.metrics.items():
    print(f"{key} : {value}")

train_runtime : 33.1545
train_samples_per_second : 60.324
train_steps_per_second : 2.111
total_flos : 18110777160000.0
train_loss : 0.03469283777023
epoch : 10.0


### Save the model for later use

In [239]:
# Save Model
print(f"[INFO] Saving model to {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

[INFO] Saving model to models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased
